In [1]:
# imports
import requests
import sys
import time
from urllib.parse import urljoin

import pandas as pd

In [2]:
# constants
PROTEINS_PLUS_URL = 'https://proteins.plus/api/v2/'
ACTIVITYFINDER = urljoin(PROTEINS_PLUS_URL, 'activityfinder/')
ACTIVITYFINDER_JOBS = urljoin(PROTEINS_PLUS_URL, 'activityfinder/jobs/')
ACTIVITYFINDER_INFO = urljoin(PROTEINS_PLUS_URL, 'activityfinder/info/')

In [3]:
#@title Utils functions to call API (unhide if you're interested)

# check server connection
try:
    response = requests.get(PROTEINS_PLUS_URL)
except requests.ConnectionError as error:
    if 'Connection refused' in str(error):
        print('WARNING: could not establish a connection to the server', file=sys.stderr)
    raise

def poll_job(job_id, poll_url, poll_interval=1, max_polls=10):
    """Poll the progress of a job

    Continuosly polls the server in regular intervals and updates the job information, especially the status.

    :param job_id: UUID of the job to poll
    :type job_id: str
    :param poll_url: URl to send the polling request to
    :type poll_url: str
    :param poll_interval: time interval between polls in seconds
    :type poll_interval: int
    :param max_polls: maximum number of times to poll before exiting
    :type max_polls: int
    :return: polled job
    :rtype: dict
    """
    job = requests.get(poll_url + job_id + '/').json()
    status = job['status']
    current_poll = 0
    while status == 'pending' or status == 'running':
        print(f'Job {job_id} is { status }')
        current_poll += 1
        if current_poll >= max_polls:
            print(f'Job {job_id} has not completed after {max_polls} polling requests' \
                  f' and {poll_interval * max_polls} seconds')
            return job
        time.sleep(poll_interval)
        job = requests.get(poll_url + job_id + '/').json()
        status = job['status']
    print(f'Job {job_id} completed with { status }')
    return job

In [4]:
def query_activityfinder_by_pdbid(pdbid, active_site_filter=True, use_cache=True):
    query = {
      "pdb_code": pdbid,
      "active_site_filter": active_site_filter,
        "use_cache" : use_cache
    }
    job_submission = requests.post(ACTIVITYFINDER, data=query).json()
    activityfinder_job = poll_job(job_submission['job_id'], ACTIVITYFINDER_JOBS)
    url = urljoin(ACTIVITYFINDER_INFO,  activityfinder_job["output_info"] + "/")
    activityfinder_info = requests.get(url)
    results = activityfinder_info.json()
    activities = None if not "ActivityValues" in results["info"].keys() else pd.DataFrame(results["info"]["ActivityValues"])
    sequence_matching_information = None if not "Sequence-Matching-Infomation-For-Assay-Target" in results["info"].keys() else pd.DataFrame(results["info"]["Sequence-Matching-Infomation-For-Assay-Target"])
    activesite_mutations = None if not "ActiveSite-Mutations-For-Target-Matches" in results["info"].keys() else pd.DataFrame(results["info"]["ActiveSite-Mutations-For-Target-Matches"])

    return activities, sequence_matching_information, activesite_mutations
pdbid = "3QRJ"
activities, sequence_matching_information, activesite_mutations = query_activityfinder_by_pdbid(pdbid, active_site_filter=True)

Job 1f3772ab-f1ba-4fd0-81e8-f6c79b9fe64c is running
Job 1f3772ab-f1ba-4fd0-81e8-f6c79b9fe64c completed with success


In [5]:
# Investigate Activities
if type(activities) != pd.DataFrame:
    print(f"We find no activities for pdb ID {pdbid}")
else:
    print(f"We find {activities.shape[0]} activity matches for pdb ID {pdbid}")
    print(f"This data is the heart of the results of the data from the ActivityFinder with the actual activity values, quality of the link and information on both the PDB structure and the ChEMBL Data")
    activities['blast_match.percent_identity'] = pd.to_numeric(activities['blast_match.percent_identity'], errors='coerce')
    print(f"The matches that we find have a maximum sequence identity of {activities['blast_match.percent_identity'].max()}")
    print(f"There are {activities['assay.chembl_id'].nunique()} unique assays in the data and we find {activities['target.chembl_id'].nunique()} unique targets")
    print(f"There are activities for {activities['pdb_ligand.ligand_id'].unique()} naomi names linked to {activities['chembl_ligand.chembl_id'].unique()}")
    activities['activity.pchembl_value'] = pd.to_numeric(activities['activity.pchembl_value'], errors='coerce')
    print(f"The average pChEMBL value of the linked ligands is {activities['activity.pchembl_value'].mean()}")
    print(activities["assay.chembl_id"].unique())

We find 23 activity matches for pdb ID 3QRJ
This data is the heart of the results of the data from the ActivityFinder with the actual activity values, quality of the link and information on both the PDB structure and the ChEMBL Data
The matches that we find have a maximum sequence identity of 100.0
There are 6 unique assays in the data and we find 2 unique targets
There are activities for ['919'] naomi names linked to ['CHEMBL1738757']
The average pChEMBL value of the linked ligands is 7.961052631578947
['CHEMBL3991603' 'CHEMBL3991604' 'CHEMBL5060015' 'CHEMBL5108948'
 'CHEMBL5108949' 'CHEMBL5133794']


In [6]:
# Investigate sequence matching information
if type(sequence_matching_information) != pd.DataFrame:
    print(f"We find no sequence matching data for pdb ID {pdbid}")
else:
    print(f"There are {sequence_matching_information.shape[0]} rows of sequence matching data for pdb ID {pdbid}")
    sequence_matching_information['blast_match.percent_identity'] = pd.to_numeric(sequence_matching_information['blast_match.percent_identity'], errors='coerce')
    print(f"We find {sequence_matching_information['blast_match.bmid'].nunique()} unique blast match ids here but {sequence_matching_information.shape[0]} rows, which is due to the fact that each row is for a specific assay")
    print(f"If you want further information on the sequence matching, you can join the data into the activity data using the blast match id and the assay like this:")
    combined_data = activities.merge(sequence_matching_information, how='left', on=[x for x in sequence_matching_information.columns.values.tolist() if x in activities.columns.values.tolist()])
    print(f"Frequently it will happen that this way you get more rows than before, this is due to the fact that different sequences can have an identical match and therefore identical blast match id and therefore are treated as identical in the activity data but not in the sequence matching data")

There are 40 rows of sequence matching data for pdb ID 3QRJ
We find 11 unique blast match ids here but 40 rows, which is due to the fact that each row is for a specific assay
If you want further information on the sequence matching, you can join the data into the activity data using the blast match id and the assay like this:
Frequently it will happen that this way you get more rows than before, this is due to the fact that different sequences can have an identical match and therefore identical blast match id and therefore are treated as identical in the activity data but not in the sequence matching data


In [7]:
# Investigate mutations
if type(activesite_mutations) != pd.DataFrame:
    print(f"We find no active site mutation data for pdb ID {pdbid}")
else:
    print("We also record all mutations that are in the active site of a ligand, defined as 6.5 Å around any heavy atom.")
    print(f"For pdb ID {pdbid}, we find {activesite_mutations.shape[0]} entries of this.")
    print(f"Please be aware that, similarly to before, the rows are repeated for each assay that is matched. There are only {activesite_mutations['quadruple.qid'].nunique()} quadruples found for the {activesite_mutations.shape[0]} rows")
    print(f"The data is reported in quadruples, which tells you exactly where the mutation is, which sequences are involved and what the values are in the primary/query sequence (PDB) and the hit sequence (ChEMBL).")

We also record all mutations that are in the active site of a ligand, defined as 6.5 Å around any heavy atom.
For pdb ID 3QRJ, we find 39 entries of this.
Please be aware that, similarly to before, the rows are repeated for each assay that is matched. There are only 23 quadruples found for the 39 rows
The data is reported in quadruples, which tells you exactly where the mutation is, which sequences are involved and what the values are in the primary/query sequence (PDB) and the hit sequence (ChEMBL).
